# Consultas SQL · cuaderno del proyecto
**Estud-IA · Análisis de Datos Básico**

Corre las celdas en orden con el botón ▶ (o Shift + Enter). Si Colab muestra una advertencia porque el cuaderno viene de GitHub, elige **«Ejecutar de todos modos»**.

1. La celda 1 prepara todo. No hay que cambiar nada.
2. La celda 2 carga tu muestra.
3. La celda 3 cuenta las filas: tu primera **cifra de control**.
4. Desde la sesión 3: tus dos consultas, más abajo.

In [ ]:
# Celda 1 · Preparar (no cambies nada)
try:
    import duckdb
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb
import re
import pandas as pd

def cargar(ruta):
    """Lee un CSV (coma o punto y coma) o un Excel. Acepta el enlace Raw o el de la página de GitHub."""
    ruta = re.sub(r"^https://github\.com/([^/]+)/([^/]+)/blob/", r"https://raw.githubusercontent.com/\1/\2/", ruta.strip())
    if ruta.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(ruta)
    for codificacion in ("utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(ruta, sep=None, engine="python", encoding=codificacion)
        except UnicodeDecodeError:
            continue
        except pd.errors.ParserError:
            raise ValueError("No pude leerlo como tabla. Revisa que el enlace sea el de tu CSV (botón Raw).") from None
    raise ValueError("No pude leer el archivo: revisa que sea un CSV")

def sql(consulta):
    """Corre una consulta SQL sobre las tablas cargadas y muestra el resultado."""
    return duckdb.sql(consulta).df()

print("Listo. Sigue con la celda 2.")

## 2. Carga tu muestra

**Opción A (recomendada).** En GitHub abre tu CSV, copia la dirección de la página (o la del botón **Raw**) y pégala entre las comillas de `ENLACE`.

**Opción B.** Deja `ENLACE = ""` y corre la celda: aparece un botón para subir el archivo desde tu computador.

Con el enlace que viene (el caso de la papelería) deben salir **8 filas y 6 columnas**.

In [ ]:
# Celda 2 · Cargar la muestra
ENLACE = "https://raw.githubusercontent.com/santiagom26cc/ejemplo-proyecto-datos-basico/main/datos/originales/ventas.csv"

if ENLACE:
    datos = cargar(ENLACE)
else:
    from google.colab import files
    subido = files.upload()
    datos = cargar(next(iter(subido)))

print(f"Cargué {len(datos)} filas y {len(datos.columns)} columnas.")
datos.head()

## 3. ¿Cuántas filas hay?

Compara con lo que escribiste en el diccionario, en «Filas en la muestra». Si no coincide, anótalo en la bitácora.

In [ ]:
# Celda 3 · Contar filas
sql("""
SELECT COUNT(*) AS filas
FROM datos
""")

## 4. Los nombres exactos de tus columnas

Cópialos de aquí. Si un nombre tiene espacios, tildes o mayúsculas, escríbelo entre **comillas dobles**: `"Fecha del hecho"`. Los valores de texto van entre **comillas simples**: `'completada'`.

In [ ]:
# Celda 4 · Columnas de la muestra
list(datos.columns)

---
# Sesión 3 · Tus dos consultas

**Consulta 1 · Mirar y filtrar.** Primero cuenta todas las filas; después cuenta las que deja tu filtro; al final mira algunas. Cambia `estado`, `'completada'` y las columnas por las de tu muestra.

Con la papelería: 8 filas en total y 7 después del filtro.

In [ ]:
# Celda 5 · Consulta 1a: filas antes del filtro
sql("""
SELECT COUNT(*) AS filas_total
FROM datos
""")

In [ ]:
# Celda 6 · Consulta 1b: filas después del filtro
sql("""
SELECT COUNT(*) AS filas_filtradas
FROM datos
WHERE estado = 'completada'
""")

In [ ]:
# Celda 7 · Consulta 1c: mirar las filas filtradas, solo las columnas que necesitas
sql("""
SELECT id_venta, producto_id, cantidad
FROM datos
WHERE estado = 'completada'
LIMIT 10
""")

**Consulta 2 · Resumir.** Cuenta cuántas filas hay en cada grupo. Cambia `estado` por la columna con la que quieres comparar (comuna, mes, categoría…).

**Cifra de control:** la suma de los grupos debe dar lo mismo que el total de la celda 5. Con la papelería: completada 7 + cancelada 1 = 8.

In [ ]:
# Celda 8 · Consulta 2: resumen por grupo
sql("""
SELECT estado, COUNT(*) AS filas
FROM datos
GROUP BY estado
ORDER BY filas DESC
""")

In [ ]:
# Celda 9 · Cifra de control: ¿la suma de los grupos da el total?
sql("""
SELECT SUM(filas) AS suma_de_los_grupos
FROM (
    SELECT estado, COUNT(*) AS filas
    FROM datos
    GROUP BY estado
)
""")

## Guarda tus consultas en GitHub

En tu repositorio, entra a `consultas/`, **Add file › Create new file**, ponle nombre (`01_mirar.sql`, `02_resumen.sql`), pega la consulta con tres comentarios arriba y haz commit con un mensaje que diga qué averiguaste:

```sql
-- Pregunta: ¿cuántas ventas se completaron?
-- Datos: datos/originales/ventas.csv (8 filas)
-- Cifra de control: 8 en total, 7 completadas
SELECT COUNT(*) AS filas_filtradas
FROM datos
WHERE estado = 'completada';
```

Mira el ejemplo resuelto en `consultas/` del repositorio de la papelería.

---
¿Algo falló? Regla de 3 y, si sigues bloqueado, un ticket en la **Cola de ayuda** con el mensaje de error copiado. Nunca pegues contraseñas ni datos personales.